# ROC Curve and Feature Importance

This file should be executed **after** running `main.ipynb`.  

## What this file does
- **ROC Curve:**  
  - Produces ROC curves for unmatched vs age-matched

- **Feature Importance:**  
  - Creates Feature Importance plots from the trained XGBoost models.  


In [ ]:
from functions import *

# load the data
df = pd.read_csv(r"../Data Cleaning/results/merged_labeled.csv", index_col="eid")
df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 2})


CAT_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Alcohol Consumption",
    "Smoking Status",
    "Antihypertensive usage",
]
NUM_FEATURES = df.columns.difference(CAT_FEATURES + ["Status", "eid"])

MAIN_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Spherical equivalent",
    "Systolic blood pressure",
    "Diastolic blood pressure",
    "Antihypertensive usage",
    "Alcohol Consumption",
    "Smoking Status",
    "BMI",
    "mRNFL thickness",
    "mGCIPL thickness",
    "Status",
]
df_main = df.loc[:, MAIN_FEATURES]
MAIN_NUM_FEATURES = df_main.columns.difference(CAT_FEATURES + ["Status", "eid"])
MAIN_CAT_FEATURES = CAT_FEATURES

input_path = r"../Data Cleaning/results/"
output_path = r"./results_combined/"

In [ ]:
os.makedirs(output_path + "final/roc_curves/pngs", exist_ok=True)


def get_rank(df1, feature, current_rank, up=True):

    rank = np.where(df1["Feature"] == feature)[0][0]

    if up:
        rankup = rank - current_rank
    else:
        rankup = current_rank - rank
    if rankup > 0:
        return f"+{rankup}"
    else:
        return f"{rankup}"


def append_roc_metrics(
    outname,
    roc_auc,
    std_auc,
    tpr_at_target,
    std_tpr_at_target,
    mroc_auc,
    mstd_auc,
    mtpr_at_target,
    mstd_tpr_at_target,
    csv_path,
):
    # Normalize path for WSL/Windows mixed environments and make sure parent folder exists.
    csv_file = Path(csv_path).expanduser()
    if not csv_file.is_absolute():
        csv_file = (Path.cwd() / csv_file).resolve()
    csv_file.parent.mkdir(parents=True, exist_ok=True)

    row = {
        "name": outname,
        "ext_mAUC": roc_auc,
        "ext_std_AUC": std_auc,
        "ext_mTPR@15%FPR": tpr_at_target,
        "ext_std_TPR": std_tpr_at_target,
        "core_mAUC": mroc_auc,
        "core_std_AUC": mstd_auc,
        "core_mTPR@15%FPR": mtpr_at_target,
        "core_std_TPR": mstd_tpr_at_target,
    }

    # Write header on first write or when file exists but is empty.
    write_header = (not csv_file.exists()) or (csv_file.stat().st_size == 0)

    pd.DataFrame([row]).to_csv(csv_file, mode="a", header=write_header, index=False)

    # Small debug line to confirm the exact target path and file size.
    print(
        f"[append_roc_metrics] wrote 1 row -> {csv_file} (size={csv_file.stat().st_size} bytes)"
    )


def plot_roc(path, name, outname="", csv_path=None):

    df1 = pd.read_csv(path + name + "_roc_curve.csv")
    df2 = pd.read_csv(path + "core - " + name + "_roc_curve.csv")

    fpr = df1["False Positive Rate"]
    tpr = df1["Mean True Positive Rate"]
    stpr = df1["Std True Positive Rate"]
    # df olması için eşit boyda olması lazımdı bu yüzden aucu birden fazla kez yazıyor, hepsi aynı değer 1.sini alıyoruz
    roc_auc = df1["ROC Score"].iloc[0]
    std_auc = df1["Std roc score"].iloc[0]
    tpr_at_target = df1["Mean tpr at target"].iloc[0]
    std_tpr_at_target = df1["Std tpr at target"].iloc[0]

    mfpr = df2["False Positive Rate"]
    mtpr = df2["Mean True Positive Rate"]
    mstpr = df2["Std True Positive Rate"]
    mroc_auc = df2["ROC Score"].iloc[0]
    mstd_auc = df2["Std roc score"].iloc[0]
    mtpr_at_target = df2["Mean tpr at target"].iloc[0]
    mstd_tpr_at_target = df2["Std tpr at target"].iloc[0]

    fig = plt.figure(figsize=(8, 6))
    plt.plot(
        fpr,
        tpr,
        label=r"Extended Feats (mAUC={} $\pm$ {})".format(roc_auc, std_auc),
        linewidth=3,
        alpha=0.7,
    )
    plt.fill_between(fpr, tpr - stpr, tpr + stpr, alpha=0.2)
    plt.plot(
        mfpr,
        mtpr,
        label=r"Core Feats (mAUC={} $\pm$ {})".format(mroc_auc, mstd_auc),
        linewidth=3,
        alpha=0.7,
    )
    plt.fill_between(mfpr, mtpr - mstpr, mtpr + mstpr, alpha=0.2)

    plt.xlabel("False Positive Rate")
    plt.ylabel("Mean True Positive Rate")
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess", linewidth=2)

    # Vertical dashed line at FPR = 0.1
    target_fpr = 0.15
    plt.axvline(
        x=target_fpr,
        color="gray",
        linestyle="--",
        linewidth=2,
        alpha=0.8,
        label=f"FPR={target_fpr}",
    )

    # Intersection points for Extended Features ROC

    plt.scatter(
        target_fpr,
        tpr_at_target,
        color="blue",
        s=300,
        marker="*",
        zorder=10,
        label=r"Extended mTPR={} $\pm$ {}".format(tpr_at_target, std_tpr_at_target),
    )

    # Intersection points for Core Features ROC
    plt.scatter(
        target_fpr,
        mtpr_at_target,
        color="tomato",
        s=300,
        marker="*",
        zorder=10,
        label=r"Core mTPR={} $\pm$ {}".format(mtpr_at_target, mstd_tpr_at_target),
    )

    plt.legend(loc="lower right")
    plt.grid(True)

    outpath = f"./results_combined/final/roc_curves/pngs/{outname}.pdf"
    os.makedirs(os.path.dirname(outpath), exist_ok=True)
    plt.savefig(outpath)
    plt.close(fig)

    if csv_path is not None:
        append_roc_metrics(
            outname,
            roc_auc,
            std_auc,
            tpr_at_target,
            std_tpr_at_target,
            mroc_auc,
            mstd_auc,
            mtpr_at_target,
            mstd_tpr_at_target,
            csv_path,
        )


def create_importance_graph(path, name1, name2, output_path):
    ad_knn_feature_importances = pd.read_csv(path + name1)
    ad_knn_feature_importances_agematched = pd.read_csv(path + name2)

    merged = ad_knn_feature_importances.merge(
        ad_knn_feature_importances_agematched,
        on="Feature",
        suffixes=("", "_agematched"),
    )

    merged_top15 = merged.sort_values("Mean Importance", ascending=False).head(15)
    merged_top15.reset_index(inplace=True, drop=True)
    merged_top15_agematched = merged.sort_values(
        "Mean Importance_agematched", ascending=False
    ).head(15)
    merged_top15_agematched.reset_index(inplace=True, drop=True)

    fig, ax = plt.subplots(figsize=(20, 10))
    sns.barplot(
        x="Mean Importance",
        y="Feature",
        data=merged_top15,
        ax=ax,
        palette="viridis",
        hue="Feature",
        legend=False,
    )

    for i, row in merged_top15.iterrows():
        label = f"{row['Mean Importance']:.3f} ({get_rank(ad_knn_feature_importances_agematched, row['Feature'], i, up=False)})"
        ax.text(
            row["Mean Importance"] + 0.002,
            i,
            label,
            va="center",
            fontsize=21,
            color="black",
            fontweight="bold",
        )

    # find x max value for both plots
    xmax = max(
        merged_top15["Mean Importance"].max(),
        merged_top15_agematched["Mean Importance_agematched"].max(),
    )

    ax.set_xlabel("Mean Importance", fontsize=25, fontweight="normal")
    ax.set_ylabel("Feature", fontsize=25, fontweight="normal")
    ax.set_yticks(ticks=np.arange(len(merged_top15["Feature"])))
    ax.set_yticklabels(merged_top15["Feature"], fontsize=27, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.xlim(0, xmax + 0.03)
    plt.tight_layout()
    plt.savefig(
        output_path + name1.split("_all_features.csv")[0] + "_feature_importances.pdf"
    )
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(20, 10))
    sns.barplot(
        x="Mean Importance_agematched",
        y="Feature",
        data=merged_top15_agematched,
        ax=ax,
        palette="viridis",
        hue="Feature",
        legend=False,
    )

    for i, row in merged_top15_agematched.iterrows():
        label = f"{row['Mean Importance_agematched']:.3f} ({get_rank(ad_knn_feature_importances, row['Feature'], i, up=True)})"
        ax.text(
            row["Mean Importance_agematched"] + 0.002,
            i,
            label,
            va="center",
            fontsize=22,
            color="black",
            fontweight="bold",
        )

    ax.set_xlabel("Mean Importance", fontsize=25, fontweight="normal")
    ax.set_ylabel("Feature", fontsize=25, fontweight="normal")
    ax.set_yticks(ticks=np.arange(len(merged_top15_agematched["Feature"])))
    ax.set_yticklabels(
        merged_top15_agematched["Feature"], fontsize=27, fontweight="bold"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.xlim(0, xmax + 0.04)
    plt.tight_layout()
    plt.savefig(
        output_path + name2.split("_all_features.csv")[0] + "_feature_importances.pdf"
    )
    plt.show()
    plt.close(fig)

    plt.close()

In [ ]:
from pathlib import Path

path = r"./results_combined/final/roc_curves/"

metrics_csv = str((Path.cwd() / path / "metrics_summary.csv").resolve())
print(f"metrics_csv -> {metrics_csv}")

# missing matched

randomfill_mm_path = path + "randomfill_mm - "
randomfill_age_matched_mm_path = path + "randomfill_age_matched_mm - "
knn_mm_path = path + "knn_mm - "
knn_age_matched_mm_path = path + "knn_age_matched_mm - "
missing_mm_path = path + "missing_mm - "
missing_age_matched_mm_path = path + "missing_age_matched_mm - "

In [ ]:
# missing matched
# missing
plot_roc(
    missing_mm_path,
    "Dementia vs Healthy",
    "Missing - mm - Dementia vs Healthy",
    csv_path=metrics_csv,
)
plot_roc(
    missing_age_matched_mm_path,
    "Dementia vs Healthy",
    "Missing - mm - Age Matched - Dementia vs Healthy",
    csv_path=metrics_csv,
)

plot_roc(
    missing_mm_path,
    "AD vs Healthy",
    "Missing - mm - AD vs Healthy",
    csv_path=metrics_csv,
)
plot_roc(
    missing_age_matched_mm_path,
    "AD vs Healthy",
    "Missing - mm - Age Matched - AD vs Healthy",
    csv_path=metrics_csv,
)

# randomfill
plot_roc(
    randomfill_mm_path,
    "Dementia vs Healthy",
    "Randomfill - mm - Dementia vs Healthy",
    csv_path=metrics_csv,
)
plot_roc(
    randomfill_age_matched_mm_path,
    "Dementia vs Healthy",
    "Randomfill - mm - Age Matched - Dementia vs Healthy",
    csv_path=metrics_csv,
)

plot_roc(
    randomfill_mm_path,
    "AD vs Healthy",
    "Randomfill - mm - AD vs Healthy",
    csv_path=metrics_csv,
)
plot_roc(
    randomfill_age_matched_mm_path,
    "AD vs Healthy",
    "Randomfill - mm - Age Matched - AD vs Healthy",
    csv_path=metrics_csv,
)

# knn
plot_roc(
    knn_mm_path,
    "Dementia vs Healthy",
    "KNN - mm - Dementia vs Healthy",
    csv_path=metrics_csv,
)
plot_roc(
    knn_age_matched_mm_path,
    "Dementia vs Healthy",
    "KNN - mm - Age Matched - Dementia vs Healthy",
    csv_path=metrics_csv,
)

plot_roc(knn_mm_path, "AD vs Healthy", "KNN - mm - AD vs Healthy", csv_path=metrics_csv)
plot_roc(
    knn_age_matched_mm_path,
    "AD vs Healthy",
    "KNN - mm - Age Matched - AD vs Healthy",
    csv_path=metrics_csv,
)

In [ ]:
path2 = r"./results_combined/final/feature_importances/"

feature_importance_path = path2 + r"paper/"
os.makedirs(feature_importance_path, exist_ok=True)

In [ ]:
# # AD vs healthy

create_importance_graph(
    path2,
    "randomfill_mm - AD vs Healthy_all_features.csv",
    "randomfill_age_matched_mm - AD vs Healthy_all_features.csv",
    feature_importance_path,
)
create_importance_graph(
    path2,
    "knn_mm - AD vs Healthy_all_features.csv",
    "knn_age_matched_mm - AD vs Healthy_all_features.csv",
    feature_importance_path,
)

# # dementia vs healthy

create_importance_graph(
    path2,
    "randomfill_mm - Dementia vs Healthy_all_features.csv",
    "randomfill_age_matched_mm - Dementia vs Healthy_all_features.csv",
    feature_importance_path,
)
create_importance_graph(
    path2,
    "knn_mm - Dementia vs Healthy_all_features.csv",
    "knn_age_matched_mm - Dementia vs Healthy_all_features.csv",
    feature_importance_path,
)

In [ ]:
import os

import pandas as pd


def export_roc_summary():
    base_path = r"./results_combined/final/roc_curves/"

    configs = [
        "missing_mm - Dementia vs Healthy",
        "missing_age_matched_mm - Dementia vs Healthy",
        "missing_mm - AD vs Healthy",
        "missing_age_matched_mm - AD vs Healthy",
        "randomfill_mm - Dementia vs Healthy",
        "randomfill_age_matched_mm - Dementia vs Healthy",
        "randomfill_mm - AD vs Healthy",
        "randomfill_age_matched_mm - AD vs Healthy",
        "knn_mm - Dementia vs Healthy",
        "knn_age_matched_mm - Dementia vs Healthy",
        "knn_mm - AD vs Healthy",
        "knn_age_matched_mm - AD vs Healthy",
    ]

    results = []

    for cfg in configs:
        extended_file = os.path.join(base_path, f"{cfg}_roc_curve.csv")
        core_file = os.path.join(base_path, f"core - {cfg}_roc_curve.csv")

        try:
            df_ext = pd.read_csv(extended_file)
            df_core = pd.read_csv(core_file)

            target = "AD vs Healthy" if "AD " in cfg else "Dementia vs Healthy"

            results.append(
                {
                    "Run Name": cfg,
                    "Setup Category": target,
                    "Extended mAUC": f"{df_ext['ROC Score'].iloc[0]:.3f} ± {df_ext['Std roc score'].iloc[0]:.3f}",
                    "Extended TPR at 15%": f"{df_ext['Mean tpr at target'].iloc[0]:.3f} ± {df_ext['Std tpr at target'].iloc[0]:.3f}",
                    "Core mAUC": f"{df_core['ROC Score'].iloc[0]:.3f} ± {df_core['Std roc score'].iloc[0]:.3f}",
                    "Core TPR at 15%": f"{df_core['Mean tpr at target'].iloc[0]:.3f} ± {df_core['Std tpr at target'].iloc[0]:.3f}",
                }
            )
        except Exception:
            pass

    final_df = pd.DataFrame(results)
    outpath = os.path.join(base_path, "roc_curves_summary.csv")
    final_df.to_csv(outpath, index=False)
    print(f"Exported summary of {len(final_df)} runs to {outpath}")
    return final_df


summary_df = export_roc_summary()
summary_df.head()